# Inspect SFT Batch Snapshots

Set `SNAPSHOT_DIR` below to the directory passed as `SFT_BATCH_SNAPSHOT_DIR`. The notebook indexes JSON dumps, renders token/loss masks, and can load the original source parquet row recorded in `debug_sample_info`.

In [ ]:
from pathlib import Path
import json
import os

import pandas as pd
from IPython.display import HTML, display

SNAPSHOT_DIR = Path(os.environ.get("SFT_BATCH_SNAPSHOT_DIR", "/tmp/sft_debug_batches")).expanduser()
SNAPSHOT_DIR

In [ ]:
def load_snapshot(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

def build_index(snapshot_dir):
    rows = []
    for path in sorted(snapshot_dir.glob("step_*_rank_*.json")):
        payload = load_snapshot(path)
        for sample in payload.get("samples", []):
            source = sample.get("source") or {}
            rows.append({
                "path": str(path),
                "global_step": payload.get("global_step"),
                "rank": payload.get("rank"),
                "dp_rank": payload.get("data_parallel_rank"),
                "local_sample_idx": sample.get("local_sample_idx"),
                "sequence_length": sample.get("sequence_length"),
                "num_loss_tokens": sample.get("num_loss_tokens"),
                "is_truncated": sample.get("is_truncated"),
                "source_file": source.get("source_file"),
                "source_row": source.get("source_row"),
                "global_row": source.get("global_row"),
            })
    return pd.DataFrame(rows)

index_df = build_index(SNAPSHOT_DIR)
index_df

In [ ]:
# Pick a sample to inspect.
ROW = 0

row = index_df.iloc[ROW]
payload = load_snapshot(Path(row.path))
sample = next(s for s in payload["samples"] if s["local_sample_idx"] == row.local_sample_idx)

print("snapshot:", row.path)
print("step/rank/sample:", row.global_step, row.rank, row.local_sample_idx)
print("source:", sample.get("source"))
print("sequence_length:", sample["sequence_length"])
print("num_loss_tokens:", sample["num_loss_tokens"])
print("loss spans:", sample["loss_mask_spans"][:20])
print("truncated:", sample["is_truncated"], "cap:", sample["truncated_to_tokens"])

In [ ]:
def html_escape(text):
    return (str(text)
        .replace("&", "&amp;")
        .replace("<", "&lt;")
        .replace(">", "&gt;"))

def render_token_table(sample, start=0, end=300):
    tokens = sample.get("tokens") or []
    rows = []
    for tok in tokens[start:end]:
        bg = "#ffe8e8" if tok.get("loss") else "#f7f7f7"
        rows.append(
            f"<tr style='background:{bg}'>"
            f"<td>{tok['idx']}</td>"
            f"<td>{tok['id']}</td>"
            f"<td>{tok.get('loss')}</td>"
            f"<td><code>{html_escape(tok.get('text', ''))}</code></td>"
            "</tr>"
        )
    html = """
    <table>
      <thead><tr><th>idx</th><th>id</th><th>loss</th><th>token</th></tr></thead>
      <tbody>{}</tbody>
    </table>
    """.format("\n".join(rows))
    display(HTML(html))

render_token_table(sample, start=0, end=300)

In [ ]:
print(sample["decoded_text"][:8000])

In [ ]:
def load_source_row(sample):
    source = sample.get("source") or {}
    source_file = source.get("source_file")
    source_row = source.get("source_row")
    if source_file is None or source_row is None:
        return None
    df = pd.read_parquet(source_file)
    return df.iloc[int(source_row)]

source_row = load_source_row(sample)
source_row